# 05 — GLM-web Probe (chat.z.ai DOM dump)
Same playbook as 04. Cells run top-down in one kernel (browser `g` shared). Run on server.

In [ ]:
import os
import zendriver
from bs4 import BeautifulSoup
g = await zendriver.start(headless=True)
await g.get('https://chat.z.ai/')
try:
    await g.main_tab.verify_cf()
except Exception as e:
    print('cf:', type(e).__name__)
html = await g.main_tab.evaluate('document.documentElement.outerHTML', await_promise=True, return_by_value=True)
soup = BeautifulSoup(html, 'html.parser')
print('== title:', soup.title.get_text(strip=True) if soup.title else None)
print('== inputs ==')
for i in soup.find_all('input'):
    print(i.get('type'), '|', (i.get('class') or [])[:4], '|', (i.get('placeholder') or '')[:50])
print('== buttons/text-clickables (first 20) ==')
n = 0
for el in soup.find_all(['button']) + soup.find_all(attrs={'role': 'button'}):
    print(el.name, (el.get('class') or [])[:4], '|', el.get_text(' ', strip=True)[:50])
    n += 1
    if n >= 20:
        break
print('== textareas:', len(soup.find_all('textarea')))
print('==MARKDOWN-SYSTEM-CLASSES==')
seen = set()
for el in soup.find_all(class_=True):
    for c in (el.get('class') or []):
        if any(k in c.lower() for k in ('markdown', 'message', 'chat')) and c not in seen:
            seen.add(c)
            print(c)
    if len(seen) > 20:
        break


In [ ]:
import asyncio as _a
btns = await g.main_tab.select_all('button')
for b in btns:
    try:
        t = (await b.get_text() if hasattr(b, 'get_text') else '')
    except Exception:
        t = ''
    if 'sign in' in (t or '').lower():
        await b.click()
        print('clicked Sign in')
        break
await _a.sleep(4)
h2 = await g.main_tab.evaluate('document.documentElement.outerHTML', await_promise=True, return_by_value=True)
s = BeautifulSoup(h2, 'html.parser')
print('== sign-in inputs ==')
for i in s.find_all('input'):
    print(i.get('type'), '|', (i.get('class') or [])[:4], '|', (i.get('placeholder') or '')[:60])
print('== sign-in buttons (first 15) ==')
n = 0
for el in s.find_all('button'):
    print((el.get('class') or [])[:4], '|', el.get_text(' ', strip=True)[:60])
    n += 1
    if n >= 15:
        break
print('== localStorage keys ==')
print(await g.main_tab.evaluate('Object.keys(localStorage)', await_promise=True, return_by_value=True))


In [ ]:
tok = os.getenv('GLM_TOKEN')
assert tok, 'export GLM_TOKEN first'
await g.main_tab.evaluate(f"localStorage.setItem('token', '{tok}')", await_promise=True, return_by_value=True)
await g.main_tab.reload()
await _a.sleep(4)
h3 = await g.main_tab.evaluate('document.documentElement.outerHTML', await_promise=True, return_by_value=True)
s3 = BeautifulSoup(h3, 'html.parser')
print('signed out (Sign in present):', 'Sign in' in s3.get_text())
tas = s3.find_all('textarea')
print('textareas:', len(tas))
for el in tas:
    print((el.get('class') or [])[:4], '|', (el.get('placeholder') or '')[:60])
print('has messageInputContainer:', bool(s3.find(class_='messageInputContainer')))
